In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType
import torch

# --- Load and Split Dataset ---
dataset = load_dataset("text", data_files={"train": "data/output_chat.txt"})["train"]
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

# --- Load Tokenizer ---
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token

# --- Tokenize ---
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
eval_dataset = eval_dataset.map(tokenize, batched=True, remove_columns=["text"])

# --- Load Base Model ---
base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_model.resize_token_embeddings(len(tokenizer))

# --- LoRA Adapter Config ---
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],  # QKV projection layer in GPT-2
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# --- Data Collator ---
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --- Training Arguments ---
training_args = TrainingArguments(
    output_dir="models/gpt2-finetuned-lora",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    save_steps=500,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
)

# --- Trainer ---
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# --- Train ---
trainer.train()

# --- Evaluate ---
results = trainer.evaluate()
print("Evaluation Results on Test Set:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

# --- Save Model & Tokenizer ---
model.save_pretrained("models/gpt2-finetuned-lora")
tokenizer.save_pretrained("models/gpt2-finetuned-lora")


Map:   0%|          | 0/7720 [00:00<?, ? examples/s]

Map:   0%|          | 0/858 [00:00<?, ? examples/s]

/home/nitin/miniconda3/envs/dl/lib/python3.12/site-packages/peft/tuners/lora/layer.py:1768: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/home/nitin/miniconda3/envs/dl/lib/python3.12/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_42210/4197872239.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.114400,4.081038
2,3.800400,3.920044
3,3.773500,3.876226


Evaluation Results on Test Set:
eval_loss: 3.8762
eval_runtime: 8.3252
eval_samples_per_second: 103.0610
eval_steps_per_second: 25.8250
epoch: 3.0000


('models/gpt2-finetuned-lora/tokenizer_config.json',
 'models/gpt2-finetuned-lora/special_tokens_map.json',
 'models/gpt2-finetuned-lora/vocab.json',
 'models/gpt2-finetuned-lora/merges.txt',
 'models/gpt2-finetuned-lora/added_tokens.json',
 'models/gpt2-finetuned-lora/tokenizer.json')

In [2]:
import math
print(f"Perplexity: {math.exp(results['eval_loss']):.2f}")

Perplexity: 48.24


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import PeftModel
from datasets import load_dataset
import torch
import math

# --- Load tokenizer and dataset ---
tokenizer = AutoTokenizer.from_pretrained("models/gpt2-finetuned-lora")
tokenizer.pad_token = tokenizer.eos_token

# Load and tokenize evaluation dataset (same as used in fine-tuning)
dataset = load_dataset("text", data_files={"test": "data/output_chat.txt"})["test"]
eval_dataset = dataset.train_test_split(test_size=0.1, seed=42)["test"]

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)

eval_dataset = eval_dataset.map(tokenize, batched=True, remove_columns=["text"])

# --- Common Config ---
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
training_args = TrainingArguments(
    output_dir="tmp_eval",
    per_device_eval_batch_size=4,
    dataloader_drop_last=False,
    report_to="none",
)

# --- Evaluate Base GPT-2 ---
base_model = AutoModelForCausalLM.from_pretrained("gpt2").to("cuda" if torch.cuda.is_available() else "cpu")
trainer_base = Trainer(
    model=base_model,
    args=training_args,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("▶️ Evaluating Base GPT-2")
base_results = trainer_base.evaluate()
print(f"Base GPT-2 Loss: {base_results['eval_loss']:.4f}")
print(f"Base GPT-2 Perplexity: {math.exp(base_results['eval_loss']):.2f}")

# --- Evaluate LoRA-Finetuned GPT-2 ---
print("\n▶️ Evaluating LoRA-Finetuned GPT-2")
base_model = AutoModelForCausalLM.from_pretrained("gpt2")
lora_model = PeftModel.from_pretrained(base_model, "models/gpt2-finetuned-lora")
lora_model = lora_model.to("cuda" if torch.cuda.is_available() else "cpu")

trainer_lora = Trainer(
    model=lora_model,
    args=training_args,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

lora_results = trainer_lora.evaluate()
print(f"LoRA-Finetuned Loss: {lora_results['eval_loss']:.4f}")
print(f"LoRA-Finetuned Perplexity: {math.exp(lora_results['eval_loss']):.2f}")


Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/858 [00:00<?, ? examples/s]

▶️ Evaluating Base GPT-2


/tmp/ipykernel_42210/3389713512.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_base = Trainer(


Base GPT-2 Loss: 6.1322
Base GPT-2 Perplexity: 460.46

▶️ Evaluating LoRA-Finetuned GPT-2


/tmp/ipykernel_42210/3389713512.py:50: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_lora = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


LoRA-Finetuned Loss: 3.8762
LoRA-Finetuned Perplexity: 48.24


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# --- Load tokenizer and base model ---
base_model_path = "gpt2"
finetuned_lora_path = "models/gpt2-finetuned-lora"

tokenizer = AutoTokenizer.from_pretrained(finetuned_lora_path)
base_model = AutoModelForCausalLM.from_pretrained(base_model_path)
model = PeftModel.from_pretrained(base_model, finetuned_lora_path)

lora_parameters = sum(p.numel() for n, p in model.named_parameters() if "lora" in n)
print(f"LoRA parameters: {lora_parameters}")

for k, v in model.state_dict().items():
    if "lora" in k:
        print(f"{k}: {v.size()}")


LoRA parameters: 294912
base_model.model.transformer.wte.weight: torch.Size([50257, 768])
base_model.model.transformer.wpe.weight: torch.Size([1024, 768])
base_model.model.transformer.h.0.ln_1.weight: torch.Size([768])
base_model.model.transformer.h.0.ln_1.bias: torch.Size([768])
base_model.model.transformer.h.0.attn.c_attn.base_layer.weight: torch.Size([768, 2304])
base_model.model.transformer.h.0.attn.c_attn.base_layer.bias: torch.Size([2304])
base_model.model.transformer.h.0.attn.c_attn.lora_A.default.weight: torch.Size([8, 768])
base_model.model.transformer.h.0.attn.c_attn.lora_B.default.weight: torch.Size([2304, 8])
base_model.model.transformer.h.0.attn.c_proj.weight: torch.Size([768, 768])
base_model.model.transformer.h.0.attn.c_proj.bias: torch.Size([768])
base_model.model.transformer.h.0.ln_2.weight: torch.Size([768])
base_model.model.transformer.h.0.ln_2.bias: torch.Size([768])
base_model.model.transformer.h.0.mlp.c_fc.weight: torch.Size([768, 3072])
base_model.model.transform

In [ ]:

model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

# --- Generation ---
prompt = "Wifey <3: hi"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=100,
        do_sample=True,
        temperature=0.9,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated Text:\n", generated_text)
